# RAG Pipeline Test


### Mount the drive


In [ ]:
# Mounting Google Drive.
from google.colab import drive

drive.mount("/content/drive")

# Move to folder within your Google Drive where to clone the repository.
%cd /content/drive/My Drive/project/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/project


# RAG pipeline tests


### Dependencies


In [2]:
!pip install torch
!pip install numpy
!pip install pillow
!pip install colpali-engine
!pip install dotenv
!pip install psutil
!pip install tqdm
!pip install aiohttp
!pip install ollama
!pip install pdf2image
!pip install lightrag-hku
!pip install flagembedding
!pip install sentence-transformers
!pip install nltk
!pip install rank-bm25
!pip install pypdf2
!pip install pytesseract
!pip install mteb
!pip install pytrec_eval

### Run


In [3]:
!pwd

/content/drive/MyDrive/project


In [4]:
%cd ms-project/src

/content/drive/MyDrive/project/ms-project/src


### Evaluate with all the knowledge bases loaded


In [ ]:
import importlib
import copy
import json
import sys, asyncio
from pathlib import Path

src = Path.cwd()
sys.path.append(str(src))

import importlib
from pipeline.rags import factory_rag
from pipeline.rags import multirag
from evaluation import evaluate

importlib.reload(factory_rag)
importlib.reload(multirag)
importlib.reload(evaluate)


# Base traditional configs
base_configs = {
    "type": "multi",
    "name": "multi",
    "configs": {
        "embedding_model": "colpali_embed",
        "generation_model": "colqwen2_ollama_gen",
        "preferred_device": "cuda",
        "traditional": {
            "type": "traditional",
            "name": "traditional",
            "configs": {
                "embedding_model": "nomic_hf_embed",
                "generation_model": "colqwen2_ollama_gen",
                "preferred_device": "cuda",
                "retrieval_method": "hybrid",
                "reranker": {
                    "name": "jina",
                    "configs": {"embedding_model": "jina_embed"},
                },
            },
        },
        "multimodal": {
            "type": "multimodal",
            "name": "multimodal",
            "configs": {
                "embedding_model": "colpali_embed",
                "generation_model": "colqwen2_ollama_gen",
                "preferred_device": "cuda",
                "chunking_strategy": "page",
                "reranker": {
                    "name": "jina",
                    "configs": {"embedding_model": "jina_embed"},
                },
            },
        },
    },
}

dynamic_fields = {
    "knowledge_base": [
        ("vidore/arxivqa_test_subsampled_beir", "arxivqa"),
        ("vidore/infovqa_test_subsampled_beir", "infovqa"),
        ("vidore/docvqa_test_subsampled_beir", "docvqa"),
        ("vidore/tabfquad_test_subsampled_beir", "tabfquad"),
        ("vidore/esg_reports_v2", "esg_reports"),
        ("vidore/economics_reports_v2", "economics_reports"),
        ("vidore/biomedical_lectures_v2", "biomedical_lectures"),
        ("beir/msmarco/dataset", "msmarco"),
        ("beir/nfcorpus/dataset", "nfcorpus"),
        ("beir/scidocs/dataset", "scidocs"),
        ("sherpa/consulting_light", "consulting_light"),
        ("sherpa/consulting", "consulting"),
    ],
    "fusion": {
        "method": [
            "normalize_average",
            "rerank_fuse",
            "max",
            "rrf",
        ],  # ["normalize_average", "rerank_fuse", "max", "rrf"],
        "norm": ["l1", "l2"],  # ["l1", "l2"]
        "rrf_k": [30],  # [30, 60, 100]
    },
}

dynamic_configs_path = Path.cwd() / "configs/dynamic_multi.json"

for kb, eval_name in dynamic_fields["knowledge_base"]:
    config = copy.deepcopy(base_configs)

    if kb.startswith("beir"):
        texts_kb = f"{kb}_texts"
        images_kb = f"{kb}_images"
        config["configs"]["traditional"]["configs"]["knowledge_base"] = texts_kb
        config["configs"]["multimodal"]["configs"]["knowledge_base"] = images_kb
        config["configs"]["knowledge_base"] = (
            images_kb  # Use any of the two, it doesn't matter they are the same except for the path
        )
    else:
        config["configs"]["traditional"]["configs"]["knowledge_base"] = kb
        config["configs"]["multimodal"]["configs"]["knowledge_base"] = kb
        config["configs"]["knowledge_base"] = kb

    config["configs"]["fusion"] = {}

    for fusion_method in dynamic_fields["fusion"]["method"]:
        for norm in dynamic_fields["fusion"]["norm"]:
            # Skip for max_l2 and rrf_l2, these ones are not impacted by normalization
            if norm == "l2" and fusion_method in ["max", "rrf"]:
                continue

            config["configs"]["fusion"]["method"] = fusion_method
            config["configs"]["fusion"]["norm"] = norm

            if fusion_method == "rrf":
                for rrf_k in dynamic_fields["fusion"]["rrf_k"]:
                    config["configs"]["fusion"]["rrf_k"] = rrf_k

                    with dynamic_configs_path.open("w") as f:
                        json.dump(config, f, indent=4)

                    complexity = ["v1", "v2"]
                    if kb in [
                        "vidore/esg_reports_v2",
                        "vidore/economics_reports_v2",
                        "vidore/biomedical_lectures_v2",
                        "sherpa/consulting_light",
                        "sherpa/consulting",
                    ]:
                        for c in complexity:
                            print(
                                f"Running with {kb} ({c}, {fusion_method}, {norm})..."
                            )
                            sys.argv = [
                                "program",
                                "--rag-configs=dynamic_multi",
                                f"--evaluation-name={eval_name}_{c}_({fusion_method}, {norm})",
                                f"--complexity={c}",
                                "--disable-generation",
                            ]
                            await evaluate.main()
                            print("End of run\n")

                    else:
                        print(f"Running with {kb} ({fusion_method}, {norm})...")
                        sys.argv = [
                            "program",
                            "--rag-configs=dynamic_multi",
                            f"--evaluation-name={eval_name}_({fusion_method}, {norm})",
                            f"--complexity=v1",
                            "--disable-generation",
                        ]
                        await evaluate.main()
                        print("End of run\n")

            else:
                with dynamic_configs_path.open("w") as f:
                    json.dump(config, f, indent=4)

                complexity = ["v1", "v2"]
                if kb in [
                    "vidore/esg_reports_v2",
                    "vidore/economics_reports_v2",
                    "vidore/biomedical_lectures_v2",
                    "sherpa/consulting_light",
                    "sherpa/consulting",
                ]:
                    for c in complexity:
                        print(f"Running with {kb} ({c}, {fusion_method}, {norm})...")
                        sys.argv = [
                            "program",
                            "--rag-configs=dynamic_multi",
                            f"--evaluation-name={eval_name}_{c}_({fusion_method}, {norm})",
                            f"--complexity={c}",
                            "--disable-generation",
                        ]
                        await evaluate.main()
                        print("End of run\n")

                else:
                    print(f"Running with {kb} ({fusion_method}, {norm})...")
                    sys.argv = [
                        "program",
                        "--rag-configs=dynamic_multi",
                        f"--evaluation-name={eval_name}_({fusion_method}, {norm})",
                        f"--complexity=v1",
                        "--disable-generation",
                    ]
                    await evaluate.main()
                    print("End of run\n")


Running with vidore/infovqa_test_subsampled_beir (max, l1)...
RAGS_DATA_DIR not set, using default: /content/drive/MyDrive/project/ms-project/src/data/rags


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json:   0%|          | 0.00/751 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

End of run

Running with sherpa/consulting (v1, max, l1)...
RAGS_DATA_DIR not set, using default: /content/drive/MyDrive/project/ms-project/src/data/rags


Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.21it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.35it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.36it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.43it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.33it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.12it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  4.39it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.48it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.42it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  4.98it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.38it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.46it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.44it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.24it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.41it/s]

Encoding t

End of run

Running with sherpa/consulting (v2, max, l1)...
RAGS_DATA_DIR not set, using default: /content/drive/MyDrive/project/ms-project/src/data/rags


Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.39it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.50it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.48it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.10it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.42it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.13it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.31it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.09it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  4.89it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.18it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.32it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.00it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  5.44it/s]

Encoding texts...: 100%|██████████| 1/1 [00:00<00:00,  8.70it/s]


End of run

